# DB7-020: Branch-specific BRB reliability

## Question and fixed hypotheses
The DB7-019 audit found that inertial confidence ranked correct versus incorrect predictions better than inertial BRB, while BRB was useful for the two EMG branches. This experiment tests that explanation without retraining the neural experts.

- **H1:** Replacing only inertial BRB reliability with calibrated inertial confidence improves net recovery relative to the original BRB.
- **H2:** Removing the disagreement input from only the inertial BRB reduces harm caused when the inertial expert is correct and EMG experts disagree.
- **Calibration control:** Also evaluate raw inertial confidence, so the effect of calibration is visible separately.

No accuracy improvement is assumed. All three arms are reported regardless of outcome; no winner is selected on test results.

## Unchanged data and models
All22 subjects, seed42, independent within-subject models; Exercise B E1 labels1–17. Inherit DB7-018: 200ms windows,10ms stride, train repetitions1/3/4/6, test2/5,13 fixed neural epochs. Saved EMG waveform (W), EMG spectrum (S) and inertial (I: ACC+gyro+mag) outputs are reused. No new signal preprocessing is applied. Original temperatures, global weights alpha, W/S BRB rules and W/S calibration remain frozen.

## Exact fitting and prediction
The new inertial BRB uses OOF repetitions1/3/4. It reuses the original crossfitted temperature values, fixes disagreement to0.5, and fits the original eight consequent beliefs using the same bounded L-BFGS-B loss and regularization. This preserves the original RIMER aggregation; it is not a new four-rule system.
OOF6 fits two monotonic scalar correctness calibrations: one for confidence and one for the new BRB. No test array is accepted by the fitting function.

At inference, calibrated class probabilities have shape N×3×17. Reliability has shape N×3. Only reliability column I changes. Effective weights are alpha×reliability, normalized over the three branches. The weighted probability sum has shape N×17; argmax supplies the gesture.

## Evaluation and limits
Compare original BRB, full confidence weighting, global weighting, inertial-only, joint SI/WSI controls, and the three new arms. Export accuracy, balanced accuracy, macro F1, NLL, Brier, ECE10, every window prediction, effective weights, reliability discrimination, gesture/repetition wrong counts, recovered/harmed cases, and initial/trained rules.
Primary uncertainty uses subjects as units: paired sign-flip Monte Carlo tests, subject-bootstrap confidence intervals and Holm correction across12 prespecified comparisons. Window McNemar is descriptive only because windows overlap95%.
This is an exploratory follow-up after inspecting the test set. Saved OOF models share training histories, so no independent nested validation is claimed. A successful result needs independent confirmation. One seed cannot establish seed robustness.

CPU only:22 new BRB heads and44 scalar calibrations; zero neural fits. The source archive SHA256 and frozen BRB replay are verified before completion is accepted.


In [ ]:
from pathlib import Path
SOURCE=Path('/kaggle/working/db7_020_source')
SOURCE.mkdir(exist_ok=True)


## brb_meta.py
Original DB7-018 implementation, copied unchanged. This supplies fixed rule matching, RIMER inference, fitting loss, calibration and weighted fusion.

In [ ]:
%%writefile /kaggle/working/db7_020_source/brb_meta.py
"""Training-only reliability fusion for DB7 W/S/I expert logits.

Labels are zero based. fit_meta accepts OOF predictions for repetitions 1/3/4/6
only. Repetition 6 is reserved for reliability calibration, not model fitting,
temperature selection or global fusion weights. The caller must supply shifts
computed against each OOF base model's own training-only signal references.

This is nested development, NOT independent meta cross-validation: OOF base
models may share training recordings. Outer test predictions never enter fit.
"""
from __future__ import annotations

import csv
import hashlib
import json
from pathlib import Path

import numpy as np
from scipy.optimize import minimize
from scipy.special import expit, logit, logsumexp, softmax

VERSION = "db7-brb-meta-v1"
EXPERTS = ("W", "S", "I")
FIT_REPS = (1, 3, 4)
CAL_REP = 6
EPS = 1e-9
HEAD_L2 = 0.01
CAL_L2 = 0.01
ALPHA_L2 = 0.005
MIN_EVENTS = 5
MAXITER = 180
RULE_BITS = np.array([[int(x) for x in f"{r:03b}"] for r in range(8)])


def _inputs(logits, shifts, y=None, repetitions=None):
    z = np.asarray(logits, dtype=np.float64)
    s = np.asarray(shifts, dtype=np.float64)
    if z.ndim != 3 or z.shape[1:] != (3, 17) or len(z) == 0:
        raise ValueError("logits must have nonempty shape [N,3,17]")
    if s.shape != z.shape[:2] or not np.isfinite(z).all() or not np.isfinite(s).all():
        raise ValueError("finite shifts [N,3] and logits required")
    if np.any((s < 0) | (s > 1)):
        raise ValueError("training-reference shifts must be in [0,1]")
    if y is None:
        return z, s
    yy = np.asarray(y)
    rr = np.asarray(repetitions)
    if yy.shape != (len(z),) or rr.shape != yy.shape:
        raise ValueError("y and repetitions must have shape [N]")
    if not np.isin(yy, np.arange(17)).all():
        raise ValueError("y must contain zero-based labels 0..16")
    if not np.isin(rr, (*FIT_REPS, CAL_REP)).all():
        raise ValueError("fit_meta accepts only training repetitions 1,3,4,6; test forbidden")
    if set(rr.tolist()) != {1, 3, 4, 6}:
        raise ValueError("all meta-fit repetitions 1/3/4 and calibration repetition 6 required")
    return z, s, yy.astype(np.int64), rr.astype(np.int64)


def _group_weights(groups):
    """Equal repetition weight; do not treat differing window counts as trials."""
    groups = np.asarray(groups)
    levels, counts = np.unique(groups, return_counts=True)
    return np.array([1.0 / (len(levels) * counts[np.searchsorted(levels, g)]) for g in groups])


def _binary_loss(target, predicted, weights):
    p = np.clip(predicted, EPS, 1 - EPS)
    return float(-np.sum(weights * (target * np.log(p) + (1 - target) * np.log1p(-p))))


def _optimization(result):
    return {"success": bool(result.success), "message": str(result.message),
            "iterations": int(result.nit), "objective": float(result.fun)}


def fit_temperatures(logits, y, groups):
    """Positive scalar temperature per expert, using supplied development rows."""
    weights = _group_weights(groups)
    temperatures, diagnostics = [], []
    for j in range(3):
        z = logits[:, j]
        def objective(theta):
            zz = z / np.exp(theta[0])
            p = softmax(zz, axis=1)
            loss = np.sum(weights * (logsumexp(zz, axis=1) - zz[np.arange(len(y)), y]))
            grad = np.sum(weights * (zz[np.arange(len(y)), y] - np.sum(p * zz, axis=1)))
            return float(loss + 0.001 * theta[0] ** 2), np.array([grad + .002 * theta[0]])
        opt = minimize(objective, [0.], jac=True, method="L-BFGS-B",
                       bounds=[(np.log(.05), np.log(20.))], options={"maxiter": MAXITER})
        if not np.isfinite(opt.fun):
            raise RuntimeError("nonfinite temperature optimization")
        temperatures.append(float(np.exp(opt.x[0])))
        diagnostics.append(_optimization(opt))
    return temperatures, diagnostics


def _probabilities(logits, temperatures):
    return softmax(logits / np.asarray(temperatures)[None, :, None], axis=2)


def indicators(probabilities, shifts):
    """Expert entropy, mean pairwise TV and precomputed training deviation."""
    p = np.asarray(probabilities, dtype=float)
    entropy = -np.sum(p * np.log(np.clip(p, EPS, 1)), axis=2) / np.log(17.)
    disagreement = np.zeros(p.shape[:2])
    for j in range(3):
        disagreement[:, j] = sum(.5 * np.abs(p[:, j] - p[:, k]).sum(1)
                                for k in range(3) if k != j) / 2
    return np.clip(np.stack([entropy, disagreement, shifts], axis=-1), 0, 1)


def rule_activations(q):
    """Product reference matching: [...,3] -> [...,8], sum exactly one."""
    q = np.asarray(q, dtype=float)
    if q.shape[-1] != 3 or not np.isfinite(q).all() or np.any((q < 0) | (q > 1)):
        raise ValueError("rule indicators must be finite [...,3] in [0,1]")
    a = np.prod(np.where(RULE_BITS, q[..., None, :], 1 - q[..., None, :]), axis=-1)
    return a / a.sum(axis=-1, keepdims=True)


def rimer_correct(activations, correct_beliefs, return_jacobian=False):
    """Analytical ER/RIMER for complete binary rule conclusions.

    A_n=prod(1-w+w*beta_n), B=prod(1-w), beta_n=(A_n-B)/(A0+A1-2B).
    Normalized activation is rule evidence weight. Returned jacobian is with
    respect to each rule's correctness belief (NOT its logit).
    """
    w = np.asarray(activations, dtype=float)
    b = np.asarray(correct_beliefs, dtype=float)
    if w.shape[-1] != 8 or b.shape != (8,):
        raise ValueError("eight activations and eight correctness beliefs required")
    if not np.isfinite(w).all() or not np.isfinite(b).all() or np.any((b < 0) | (b > 1)):
        raise ValueError("invalid ER beliefs")
    if np.any((w < 0) | (w > 1)) or not np.allclose(w.sum(-1), 1):
        raise ValueError("ER activations must be normalized")
    f1 = 1 - w + w * b
    f0 = 1 - w + w * (1 - b)
    a1, a0, bb = f1.prod(-1), f0.prod(-1), (1 - w).prod(-1)
    denom = a1 + a0 - 2 * bb
    if np.any(denom <= 0):
        raise FloatingPointError("degenerate ER normalization")
    p = np.clip((a1 - bb) / denom, 0, 1)
    if not return_jacobian:
        return p
    # Product excluding each factor handles exact zero factors at rule vertices.
    da1 = np.stack([w[..., k] * np.delete(f1, k, axis=-1).prod(-1) for k in range(8)], -1)
    da0 = -np.stack([w[..., k] * np.delete(f0, k, axis=-1).prod(-1) for k in range(8)], -1)
    jac = (da1 * denom[..., None] - (a1 - bb)[..., None] * (da1 + da0)) / denom[..., None] ** 2
    return p, jac


def _fit_alpha(p, y, weights):
    true_p = p[np.arange(len(p))[:, None], np.arange(3)[None, :], y[:, None]]
    def objective(theta):
        alpha = softmax(theta)
        mixture = np.clip(true_p @ alpha, EPS, 1)
        loss = -np.sum(weights * np.log(mixture)) + ALPHA_L2 * np.sum(theta ** 2)
        da = -np.sum(weights[:, None] * true_p / mixture[:, None], axis=0)
        grad = alpha * (da - alpha @ da) + 2 * ALPHA_L2 * theta
        return float(loss), grad
    opt = minimize(objective, np.zeros(3), jac=True, method="L-BFGS-B",
                   bounds=[(-6, 6)] * 3, options={"maxiter": MAXITER})
    return softmax(opt.x).tolist(), _optimization(opt)


def _raw_head(head, q):
    if head["kind"] == "constant":
        return np.full(len(q), head["value"])
    if head["kind"] == "logistic":
        return expit(np.column_stack([np.ones(len(q)), q - .5]) @ np.asarray(head["parameters"]))
    qq = q.copy()
    if head["kind"] == "brb_no_shift":
        qq[:, 2] = .5
    a = rule_activations(qq)
    beliefs = expit(head["parameters"])
    return a @ beliefs if head["kind"] == "sugeno" else rimer_correct(a, beliefs)


def _fit_head(kind, q, target, weights):
    prior = float((np.sum(target) + .5) / (len(target) + 1))
    if min(int(target.sum()), int((1 - target).sum())) < MIN_EVENTS:
        return {"kind": "constant", "requested_kind": kind, "value": prior,
                "fallback": "fewer than five correct or incorrect examples"}
    if kind == "logistic":
        x = np.column_stack([np.ones(len(q)), q - .5])
        center = np.array([logit(prior), 0, 0, 0])
        def objective(theta):
            raw = expit(x @ theta)
            loss = _binary_loss(target, raw, weights) + HEAD_L2 * np.mean((theta - center) ** 2)
            grad = x.T @ (weights * (raw - target)) + 2 * HEAD_L2 * (theta - center) / len(theta)
            return loss, grad
    else:
        qq = q.copy()
        if kind == "brb_no_shift":
            qq[:, 2] = .5
        a = rule_activations(qq)
        center = np.full(8, logit(prior))
        def objective(theta):
            beliefs = expit(theta)
            if kind == "sugeno":
                raw, jac = a @ beliefs, a
            else:
                raw, jac = rimer_correct(a, beliefs, return_jacobian=True)
            pp = np.clip(raw, EPS, 1 - EPS)
            derivative = weights * (pp - target) / (pp * (1 - pp))
            grad = (derivative @ jac) * beliefs * (1 - beliefs)
            loss = _binary_loss(target, pp, weights) + HEAD_L2 * np.mean((theta - center) ** 2)
            grad += 2 * HEAD_L2 * (theta - center) / len(theta)
            return loss, grad
    opt = minimize(objective, center, jac=True, method="L-BFGS-B", bounds=[(-10, 10)] * len(center),
                   options={"maxiter": MAXITER, "ftol": 1e-9})
    if not np.isfinite(opt.fun) or not np.isfinite(opt.x).all():
        raise RuntimeError(f"nonfinite {kind} fit")
    return {"kind": kind, "parameters": opt.x.tolist(), "optimization": _optimization(opt), "prior": prior}


def _fit_calibration(raw, target):
    """Monotone logit-affine reliability calibration on repetition 6 only."""
    if len(target) < 20:
        return {"slope": 1., "intercept": 0., "fallback": "fewer than 20 calibration rows"}
    x = logit(np.clip(raw, 1e-5, 1 - 1e-5))
    sparse = min(int(target.sum()), int((1 - target).sum())) < MIN_EVENTS
    def objective(theta):
        slope, intercept = theta
        p = expit(slope * x + intercept)
        weights = np.full(len(target), 1 / len(target))
        loss = _binary_loss(target, p, weights) + CAL_L2 * ((slope - 1) ** 2 + intercept ** 2)
        residual = p - target
        grad = np.array([np.mean(residual * x) + 2 * CAL_L2 * (slope - 1),
                         np.mean(residual) + 2 * CAL_L2 * intercept])
        return loss, grad
    opt = minimize(objective, [1., 0.], jac=True, method="L-BFGS-B",
                   bounds=[(1., 1.) if sparse else (0., 5.), (-8., 8.)], options={"maxiter": MAXITER})
    return {"slope": float(opt.x[0]), "intercept": float(opt.x[1]), "optimization": _optimization(opt),
            "fallback": "intercept-only: fewer than five events in one outcome" if sparse else None}


def _calibrated(raw, calibration):
    return expit(calibration["slope"] * logit(np.clip(raw, 1e-5, 1 - 1e-5)) + calibration["intercept"])


def _weights(alpha, reliability):
    unnormalized = np.asarray(alpha)[None, :] * np.clip(reliability, 0, 1)
    denominator = unnormalized.sum(1, keepdims=True)
    return np.divide(unnormalized, denominator, out=np.broadcast_to(alpha, unnormalized.shape).copy(), where=denominator > EPS)


def predict_diagnostics(model, logits, shifts):
    z, s = _inputs(logits, shifts)
    if model.get("version") != VERSION:
        raise ValueError("unsupported meta model version")
    p = _probabilities(z, model["temperatures"])
    q = indicators(p, s)
    reliabilities = {"confidence_weight": p.max(2)}
    raw_reliabilities = {}
    for name, heads in model["heads"].items():
        raw = np.column_stack([_raw_head(heads[j], q[:, j]) for j in range(3)])
        raw_reliabilities[name] = raw
        reliabilities[name] = np.column_stack([_calibrated(raw[:, j], model["reliability_calibrations"][name][j]) for j in range(3)])
    weights = {name: _weights(model["alpha"], r) for name, r in reliabilities.items()}
    return {"expert_probabilities": p, "indicators": q, "raw_reliabilities": raw_reliabilities,
            "reliabilities": reliabilities, "weights": weights, "rule_activations": rule_activations(q)}


def predict_meta(model, logits, shifts):
    d = predict_diagnostics(model, logits, shifts)
    p = d["expert_probabilities"]
    result = {f"expert_{name.lower()}": p[:, j] for j, name in enumerate(EXPERTS)}
    result["mean"] = p.mean(1)
    result["global_weight"] = np.sum(p * np.asarray(model["alpha"])[None, :, None], axis=1)
    result.update({name: np.sum(p * w[:, :, None], axis=1) for name, w in d["weights"].items()})
    return result


def _metrics(y, p):
    confidence, predicted = p.max(1), p.argmax(1)
    correct = predicted == y
    onehot = np.eye(p.shape[1])[y]
    ece = 0.
    for low in np.arange(0, 1, .1):
        mask = (confidence >= low) & (confidence < low + .1 if low < .9 else confidence <= 1)
        if mask.any():
            ece += mask.mean() * abs(correct[mask].mean() - confidence[mask].mean())
    return {"rows": int(len(y)), "accuracy": float(correct.mean()),
            "nll": float(-np.log(np.clip(p[np.arange(len(y)), y], EPS, 1)).mean()),
            "brier": float(np.sum((p - onehot) ** 2, axis=1).mean()), "ece10": float(ece)}


def _write_csv(path, rows):
    if not rows:
        return
    with path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


def fit_meta(logits, y, repetitions, shifts, output: Path, metadata=None):
    """Fit and save a JSON-serializable meta model; never supply test rows.

    1/3/4: final temperatures, alpha, reliability heads. Head training inputs use
    leave-one-repetition crossfitted temperatures. 6: monotone scalar reliability
    calibration only. Final temperature is frozen before looking at rep 6.
    """
    z, s, yy, rr = _inputs(logits, shifts, y, repetitions)
    fit = np.isin(rr, FIT_REPS)
    cal = rr == CAL_REP
    temperatures, tdiag = fit_temperatures(z[fit], yy[fit], rr[fit])
    p_final = _probabilities(z, temperatures)
    p_crossfit = np.zeros_like(z[fit])
    crossfits = []
    for held in FIT_REPS:
        inner_train = fit & (rr != held)
        temp, opt = fit_temperatures(z[inner_train], yy[inner_train], rr[inner_train])
        p_crossfit[rr[fit] == held] = _probabilities(z[fit & (rr == held)], temp)
        crossfits.append({"held_repetition": held, "fit_repetitions": sorted(set(rr[inner_train].tolist())),
                          "temperatures": temp, "optimization": opt})
    q_fit = indicators(p_crossfit, s[fit])
    q_cal = indicators(p_final[cal], s[cal])
    # Positive temperature cannot change an expert's class argmax.
    correctness_fit = (z[fit].argmax(2) == yy[fit, None]).astype(float)
    correctness_cal = (z[cal].argmax(2) == yy[cal, None]).astype(float)
    weights_fit = _group_weights(rr[fit])
    alpha, adiag = _fit_alpha(p_crossfit, yy[fit], weights_fit)
    model = {"version": VERSION, "expert_order": list(EXPERTS), "num_classes": 17,
             "temperatures": temperatures, "alpha": alpha, "heads": {}, "reliability_calibrations": {},
             "metadata": metadata or {}, "provenance": {
                 "meta_fit_repetitions": list(FIT_REPS), "reliability_calibration_repetition": CAL_REP,
                 "outer_test_repetitions_forbidden_at_fit": [2, 5], "fit_rows": int(fit.sum()), "calibration_rows": int(cal.sum()),
                 "counts_per_repetition": {str(r): int((rr == r).sum()) for r in sorted(set(rr.tolist()))},
                 "temperatures_crossfit": crossfits, "final_temperature_optimization": tdiag, "alpha_optimization": adiag,
                 "alpha_fit_input": "crossfitted-temperature probabilities from repetitions 1/3/4",
                 "fit_digest": hashlib.sha256(z[fit].tobytes() + yy[fit].tobytes() + s[fit].tobytes() + rr[fit].tobytes()).hexdigest(),
                 "calibration_digest": hashlib.sha256(z[cal].tobytes() + yy[cal].tobytes() + s[cal].tobytes()).hexdigest(),
                 "fixed_hyperparameters": {"head_l2": HEAD_L2, "calibration_l2": CAL_L2, "alpha_l2": ALPHA_L2, "min_events": MIN_EVENTS},
                 "rule_inputs": ["normalized_entropy", "mean_pairwise_total_variation", "training_reference_deviation"],
                 "caveat": "internal development; shared base-model training histories mean this is not independent meta CV"}}
    support_rows, rule_rows, reliability_rows, initial_rows = [], [], [], []
    for name in ("logistic", "sugeno", "brb", "brb_no_shift"):
        model["heads"][name], model["reliability_calibrations"][name] = [], []
        for j, expert in enumerate(EXPERTS):
            head = _fit_head(name, q_fit[:, j], correctness_fit[:, j], weights_fit)
            raw_cal = _raw_head(head, q_cal[:, j])
            calibration = _fit_calibration(raw_cal, correctness_cal[:, j])
            model["heads"][name].append(head)
            model["reliability_calibrations"][name].append(calibration)
            for stage, values in [("before", raw_cal), ("after", _calibrated(raw_cal, calibration))]:
                reliability_rows.append({"method": name, "expert": expert, "stage": stage, "repetition": 6,
                    "scope": "calibration fitting rows; not independent evaluation", "rows": len(raw_cal),
                    "observed_correctness": float(correctness_cal[:, j].mean()), "mean_reliability": float(values.mean()),
                    "brier_binary": float(np.mean((values - correctness_cal[:, j]) ** 2)),
                    "nll_binary": _binary_loss(correctness_cal[:, j], values, np.full(len(values), 1 / len(values)))})
            if name != "logistic":
                qq = q_fit[:, j].copy()
                if name == "brb_no_shift":
                    qq[:, 2] = .5
                a = rule_activations(qq)
                beliefs = np.full(8, head["value"]) if head["kind"] == "constant" else expit(head["parameters"])
                for rule, bits in enumerate(RULE_BITS):
                    initial = float(head.get("prior", head.get("value")))
                    initial_rows.append({"method": name, "expert": expert, "rule": rule,
                        "uncertainty": int(bits[0]), "disagreement": int(bits[1]), "deviation": int(bits[2]),
                        "belief_incorrect": 1-initial, "belief_correct": initial,
                        "initial_logit": float(logit(initial)), "rule_weight": 1.0,
                        "antecedent_weights": "1,1,1", "reference_values": "0,1"})
                    rule_rows.append({"method": name, "expert": expert, "rule": rule,
                        "uncertainty": int(bits[0]), "disagreement": int(bits[1]), "deviation": int(bits[2]),
                        "belief_incorrect": float(1 - beliefs[rule]), "belief_correct": float(beliefs[rule])})
                    for rep in FIT_REPS:
                        mask = rr[fit] == rep
                        mass = a[mask, rule]
                        support_rows.append({"method": name, "expert": expert, "rule": rule, "repetition": rep,
                            "window_count": int(mask.sum()), "activation_mass": float(mass.sum()),
                            "mean_activation": float(mass.mean()), "windows_activation_above_0_1": int((mass > .1).sum()),
                            "effective_windows_kish_correlated_not_trials": float(mass.sum() ** 2 / max(np.sum(mass ** 2), EPS)),
                            "weighted_correctness": float(mass @ correctness_fit[mask, j] / max(mass.sum(), EPS))})
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    (output / "meta_model.json").write_text(json.dumps(model, indent=2, allow_nan=False), encoding="utf-8")
    (output / "meta_provenance.json").write_text(json.dumps(model["provenance"], indent=2, allow_nan=False), encoding="utf-8")
    _write_csv(output / "meta_rule_initial.csv", initial_rows)
    _write_csv(output / "meta_rule_conclusions.csv", rule_rows)
    _write_csv(output / "meta_rule_support_per_repetition.csv", support_rows)
    _write_csv(output / "meta_reliability_calibration.csv", reliability_rows)
    pred = predict_meta(model, z, s)
    metrics = [{"method": name, "repetition": int(rep),
                "scope": "meta-head development fit" if rep in FIT_REPS else "reliability calibration fit",
                **_metrics(yy[rr == rep], pp[rr == rep])}
               for name, pp in pred.items() for rep in sorted(set(rr.tolist()))]
    _write_csv(output / "meta_development_metrics.csv", metrics)
    return model


## branch_reliability.py
fit_changes accepts only OOF inputs. predict_changes is label-free. evaluate_subject exports fixed-arm outcomes. summarize uses subject-level uncertainty; main verifies source provenance and all22 subjects.

In [ ]:
%%writefile /kaggle/working/db7_020_source/branch_reliability.py
"""DB7-020 fixed branch-specific reliability comparison; no neural retraining."""
from pathlib import Path
import io, json, hashlib, zipfile, shutil
import numpy as np
import pandas as pd
from scipy.special import expit
from scipy.stats import binomtest, rankdata
import brb_meta as b

SOURCE_SHA = '65cf7bf7b914e0eb2f4178e96cffc74d27b6d42dacd694a6eaac42c4aec54eb0'
NEW_ARMS = ('hybrid_confidence', 'hybrid_calibrated_confidence', 'inertial_brb_no_disagreement')


def fit_changes(model, logits, labels, repetitions, shifts):
    """Only training-side OOF arrays enter this function. Test inputs are absent.

    Keep temperatures, alpha and W/S reliability heads frozen. Fit one inertial
    BRB using the source temperature crossfits on reps1/3/4, with disagreement
    fixed to0.5. Calibrate its output and inertial confidence on OOF6 only.
    """
    z,s,y,r = b._inputs(logits, shifts, labels, repetitions)
    assert set(r.tolist()) == {1,3,4,6}
    fit=np.isin(r,[1,3,4]); cal=r==6
    pc=np.zeros_like(z[fit]); rf=r[fit]
    for fold in model['provenance']['temperatures_crossfit']:
        held=fold['held_repetition']; mask=rf==held
        assert held in (1,3,4) and held not in fold['fit_repetitions']
        pc[mask]=b._probabilities(z[fit][mask],fold['temperatures'])
    assert np.allclose(pc.sum(2),1)
    q=b.indicators(pc,s[fit])[:,2].copy(); q[:,1]=.5
    correct=(z[fit,2].argmax(1)==y[fit]).astype(float)
    head=b._fit_head('brb',q,correct,b._group_weights(rf))
    pcal=b._probabilities(z[cal],model['temperatures'])
    qcal=b.indicators(pcal,s[cal])[:,2].copy(); qcal[:,1]=.5
    target=(z[cal,2].argmax(1)==y[cal]).astype(float)
    return dict(inertial_no_disagreement=head,
        no_disagreement_calibration=b._fit_calibration(b._raw_head(head,qcal),target),
        confidence_calibration=b._fit_calibration(pcal[:,2].max(1),target),
        fit_repetitions=[1,3,4],calibration_repetition=6,
        fit_sha256=hashlib.sha256(z.tobytes()+y.tobytes()+r.tobytes()+s.tobytes()).hexdigest(),
        fixed_alpha=model['alpha'],fixed_temperatures=model['temperatures'])


def predict_changes(model, change, logits, shifts):
    """Prediction is label-free. Each arm replaces only inertial reliability."""
    d=b.predict_diagnostics(model,logits,shifts); p=d['expert_probabilities']
    base=d['reliabilities']['brb']; conf=p[:,2].max(1)
    q=d['indicators'][:,2].copy(); q[:,1]=.5
    values=[conf,b._calibrated(conf,change['confidence_calibration']),
        b._calibrated(b._raw_head(change['inertial_no_disagreement'],q),change['no_disagreement_calibration'])]
    predictions=b.predict_meta(model,logits,shifts)
    weights={}; reliability={}
    for name,value in zip(NEW_ARMS,values):
        r=base.copy();r[:,2]=value; w=b._weights(model['alpha'],r)
        predictions[name]=(p*w[:,:,None]).sum(1);weights[name]=w;reliability[name]=r
    return predictions,weights,reliability


def evaluate_subject(subject,bundle,model,metadata,out):
    changes=fit_changes(model,bundle['oof_logits'],bundle['oof_y'],bundle['oof_repetitions'],bundle['oof_shifts'])
    folder=out/f'S{subject:02d}';folder.mkdir(parents=True,exist_ok=True)
    (folder/'fitted_changes.json').write_text(json.dumps(changes,indent=2),encoding='utf-8')
    rules=[]
    for branch,head in [('W',model['heads']['brb'][0]),('S',model['heads']['brb'][1]),('I_original',model['heads']['brb'][2]),('I_no_disagreement',changes['inertial_no_disagreement'])]:
        beliefs=np.full(8,head['value']) if head['kind']=='constant' else expit(head['parameters'])
        for k,bits in enumerate(b.RULE_BITS):
            for stage,v in [('initial',head.get('prior',head.get('value'))),('trained',beliefs[k])]:
                rules.append(dict(branch=branch,rule=k,stage=stage,entropy_reference=int(bits[0]),disagreement_reference=int(bits[1]),shift_reference=int(bits[2]),belief_correct=v,belief_incorrect=1-v,disagreement_fixed_half=branch=='I_no_disagreement'))
    pd.DataFrame(rules).to_csv(folder/'rules_initial_trained.csv',index=False)
    pred,weights,reliability=predict_changes(model,changes,bundle['test_logits'],bundle['test_shifts'])
    # Controls are saved class probabilities, not logits; enforce this contract.
    for arm,key in [('control_si','control_si'),('control_wsi','control_wsi')]:
        p=bundle[key];assert np.allclose(p.sum(1),1,atol=1e-5) and (p>=0).all();pred[arm]=p
    y=bundle['test_y']
    diag=b.predict_diagnostics(model,bundle['test_logits'],bundle['test_shifts'])
    ep=diag['expert_probabilities']; correct_i=ep[:,2].argmax(1)==y
    disagreement=(ep.argmax(2)!=ep.argmax(2)[:,[0]]).any(1)
    scores={'original_brb':diag['reliabilities']['brb'][:,2],'confidence':ep[:,2].max(1),**{k:v[:,2] for k,v in reliability.items()}}
    relrows=[]
    for name,score in scores.items():
        for subset,mask in [('all',np.ones(len(y),bool)),('disagreement',disagreement)]:
            yy=correct_i[mask];ss=score[mask];n=int(yy.sum());m=int((~yy).sum())
            auc=float((rankdata(ss)[yy].sum()-n*(n+1)/2)/(n*m)) if n*m else np.nan
            relrows.append(dict(subject=subject,estimator=name,subset=subset,rows=len(yy),accuracy=yy.mean(),mean_reliability=ss.mean(),correctness_auc=auc,brier=np.mean((ss-yy)**2)))
    pd.DataFrame(relrows).to_csv(folder/'inertial_reliability.csv',index=False)
    assert np.array_equal(metadata.gesture.to_numpy()-1,y)
    methods=['brb','confidence_weight','global_weight','expert_i','control_si','control_wsi',*NEW_ARMS]
    rows=[];pairs=[];trace=metadata.copy();trace['subject']=subject
    for arm in methods:
        p=pred[arm];assert np.isfinite(p).all() and np.allclose(p.sum(1),1)
        correct=p.argmax(1)==y
        recalls=[correct[y==c].mean() for c in range(17)]
        f1=[]
        for c in range(17):
            tp=((p.argmax(1)==c)&(y==c)).sum();fp=((p.argmax(1)==c)&(y!=c)).sum();fn=((p.argmax(1)!=c)&(y==c)).sum()
            f1.append(2*tp/max(2*tp+fp+fn,1))
        rows.append(dict(subject=subject,arm=arm,balanced_accuracy=np.mean(recalls),macro_f1=np.mean(f1),**b._metrics(y,p)))
        trace[arm+'_prediction']=p.argmax(1)+1;trace[arm+'_correct']=correct
        for ref in ['brb','confidence_weight','global_weight','expert_i','control_si']:
            rc=pred[ref].argmax(1)==y; recovered=int((correct&~rc).sum());harmed=int((~correct&rc).sum())
            pairs.append(dict(subject=subject,arm=arm,reference=ref,recovered=recovered,harmed=harmed,net=recovered-harmed,
                window_mcnemar_p_descriptive=binomtest(recovered,recovered+harmed,.5).pvalue if recovered+harmed else 1.))
        if arm in weights:
            for j,branch in enumerate(['W','S','I']):
                trace[arm+'_'+branch+'_weight']=weights[arm][:,j]
                trace[arm+'_'+branch+'_reliability']=reliability[arm][:,j]
    trace.to_csv(folder/'test_trace.csv.gz',index=False,compression='gzip')
    grouped=[]
    for (gesture,rep),g in trace.groupby(['gesture','native_repetition']):
        for arm in methods:grouped.append(dict(subject=subject,gesture=gesture,repetition=rep,arm=arm,windows=len(g),wrong=int((~g[arm+'_correct']).sum()),accuracy=g[arm+'_correct'].mean()))
    pd.DataFrame(grouped).to_csv(folder/'gesture_repetition.csv',index=False)
    return rows,pairs


def summarize(rows,pairs,out):
    df=pd.DataFrame(rows);df.to_csv(out/'subject_metrics.csv',index=False)
    pd.DataFrame(pairs).to_csv(out/'recovery_harm.csv',index=False)
    summary=df.groupby('arm')[['accuracy','balanced_accuracy','macro_f1','nll','brier','ece10']].mean().sort_values('accuracy',ascending=False)
    summary.to_csv(out/'mean_subject_metrics.csv')
    for name in ['inertial_reliability','gesture_repetition']:
        pd.concat([pd.read_csv(p) for p in sorted(out.glob('S*/'+name+'.csv'))],ignore_index=True).to_csv(out/(name+'.csv'),index=False)
    # Subject is the resampling unit. Never treat 95%-overlapping windows as iid.
    pivot=df.pivot(index='subject',columns='arm',values='accuracy');rng=np.random.default_rng(42020);tests=[]
    for arm in NEW_ARMS:
        for ref in ['brb','confidence_weight','global_weight','control_si']:
            delta=(pivot[arm]-pivot[ref]).to_numpy();n=len(delta)
            observed=abs(delta.mean());count=0;draws=100000
            for _ in range(100):
                signs=rng.choice([-1,1],size=(1000,n));count+=int((abs((signs*delta).mean(1))>=observed-1e-15).sum())
            boots=delta[rng.integers(n,size=(20000,n))].mean(1)
            tests.append(dict(arm=arm,reference=ref,n_subjects=n,mean_difference_pp=100*delta.mean(),ci_low_pp=100*np.quantile(boots,.025),ci_high_pp=100*np.quantile(boots,.975),p_subject_signflip=(count+1)/(draws+1)))
    order=np.argsort([r['p_subject_signflip'] for r in tests]);running=0
    for rank,index in enumerate(order):
        running=max(running,min(1,tests[index]['p_subject_signflip']*(len(tests)-rank)));tests[index]['p_holm']=running
    pd.DataFrame(tests).to_csv(out/'subject_significance.csv',index=False)
    import matplotlib;matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    ax=pivot[['brb','confidence_weight',*NEW_ARMS]].mul(100).plot(figsize=(12,5),marker='o');ax.set_ylabel('Accuracy (%)');ax.figure.tight_layout();ax.figure.savefig(out/'subject_accuracy.png',dpi=150);plt.close(ax.figure)
    report='''# DB7-020 branch-specific reliability comparison
All22 subjects, seed42; frozen DB7-018 neural experts, temperatures and global weights.
Only inertial reliability changes; W/S BRBs remain identical. Three fixed new arms:
raw inertial confidence; calibrated inertial confidence; inertial BRB with disagreement fixed to0.5.
The raw-confidence arm isolates the additional effect of correctness calibration.
No neural training, no test selection, no promised improvement.

Head fitting uses OOF1/3/4; monotonic correctness calibration uses OOF6.
No independent nested validation is claimed: saved OOF models share training histories.
This is an exploratory follow-up to inspected test results, with one seed. Subject-bootstrap
intervals and paired subject sign-flip tests (Holm across12 comparisons) quantify between-subject
variation, not seed variability. Window McNemar p-values are descriptive because windows overlap.
The no-disagreement arm retains the original eight-rule RIMER parameterization with the disagreement
input clamped to0.5; it is not a new four-rule ER system. Initial and trained rules are exported.
A positive result must be confirmed using a prespecified independent protocol before claims of superiority.

Mean subject metrics (accuracy as fraction):
'''
    (out/'REPORT.md').write_text(report+'\n```\n'+summary.to_string()+'\n```\n',encoding='utf-8')


def main():
    out=Path('/kaggle/working/db7_020_branch_reliability');out.mkdir(exist_ok=True)
    sources=list(Path('/kaggle/input').rglob('db7_brb_full_20260915T013327619764Z.zip'));assert len(sources)==1
    h=hashlib.sha256()
    with sources[0].open('rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''):h.update(block)
    assert h.hexdigest()==SOURCE_SHA
    rows=[];pairs=[];subjects=[]
    with zipfile.ZipFile(sources[0]) as z:
        done=json.loads(z.read('completion.json'));assert done['success'] and done['neural_fits']==374 and done['meta_completed']==22
        names=[n for n in z.namelist() if n.startswith('full/') and n.endswith('/subject_bundle.npz')];assert len(names)==22
        for name in sorted(names):
            base=name.rsplit('/',1)[0];model=json.loads(z.read(base+'/meta/meta_model.json'));subject=int(model['metadata']['subject'])
            with np.load(io.BytesIO(z.read(name)),allow_pickle=False) as a:bundle={k:a[k] for k in a.files}
            metadata=pd.read_csv(io.BytesIO(z.read(base+'/test_metadata.csv')))
            rr,pp=evaluate_subject(subject,bundle,model,metadata,out)
            original=pd.read_csv(io.BytesIO(z.read(base+'/results/method_metrics.csv'))).set_index('method')
            assert np.isclose(next(x['accuracy'] for x in rr if x['arm']=='brb'),original.loc['brb','accuracy'],atol=1e-12,rtol=0)
            rows.extend(rr);pairs.extend(pp);subjects.append(subject);print('COMPLETED SUBJECT',subject,flush=True)
    assert sorted(subjects)==list(range(1,23));summarize(rows,pairs,out)
    done=dict(experiment_id='DB7-020',success=True,subjects=sorted(subjects),source_archive_sha256=SOURCE_SHA,neural_fits=0,new_brb_heads=22,new_scalar_calibrations=44,test_used_for_fitting=False)
    (out/'completion.json').write_text(json.dumps(done,indent=2));shutil.make_archive(str(out),'zip',out)
    print(json.dumps(done),flush=True)

if __name__=='__main__':main()


## Run the controlled comparison
Missing or altered source data stops the notebook. There is no neural retraining fallback.

In [ ]:
import subprocess,sys
subprocess.run([sys.executable,str(SOURCE/'branch_reliability.py')],check=True)


In [ ]:
from IPython.display import display,Markdown,Image
import pandas as pd
OUT=Path('/kaggle/working/db7_020_branch_reliability')
display(Markdown((OUT/'REPORT.md').read_text()))
display(pd.read_csv(OUT/'subject_significance.csv'))
display(Image(filename=str(OUT/'subject_accuracy.png')))
